# Day 1: From "It Works on My Laptop" to "Ready for the Public"

**Today's goal:** look at our SupportDesk agent the way a safety inspector looks at a restaurant *before* opening day, and write down everything that must be true before real customers arrive.

## The big analogy: opening a restaurant

Imagine you cook brilliant food at home. Friends love it. Does that mean you can open a restaurant tomorrow?

No. A restaurant also needs a front door with a bouncer, a fridge that survives a power cut, a rule for how long to keep leftovers, a fire plan, and a speed target for how fast food reaches the table. **The cooking is the same. Everything around the cooking is what makes it a business.**

Our agent is the same:

| In the restaurant | In our agent (SupportDesk) | Today's word |
| --- | --- | --- |
| Your cooking | The agent that answers correctly (you built this in L2) | the *prototype* |
| "Food arrives in 15 minutes" | "Answer within 30 seconds" | **NFR** |
| Front door + bouncer | `POST /ask` + API key + size limit | **API boundary** |
| Airport-style security check | Checking user text, model choices and answers | **Trust boundary** |
| Hotel front desk with room keys | Saved conversations with a safe `thread_id` | **State** |
| Fuse box + backup generator | Timeout + fallback for every outside call | **Failure domain** |
| Fire-safety plan | A list of "what can go wrong, what stops it" | **Risk register** |

## How to use this notebook
1. Run each cell from top to bottom: click a cell, then press **Shift + Enter**.
2. Read the short story above each code cell first. Then run the code and compare.
3. Nothing here calls an AI model. It runs offline in a few seconds.
4. In **Step 6** you will *predict* before you run. Predicting is how you find out whether you really understand.


## Step 0: Setup

We bring in two tools:
- `TypedDict` says what a "clipboard" looks like (you will see why in Step 3).
- `langgraph` lets us run our checks one after another, like stations on a conveyor belt.

(We also import `copy` and `Counter` for the experiments at the end.)


In [1]:
import copy
from collections import Counter
from typing import TypedDict

from langgraph.graph import END, START, StateGraph

print("Setup done. Ready to inspect SupportDesk.")

Setup done. Ready to inspect SupportDesk.


## Step 1: The inspection form (we call it a *spec*)

A health inspector arrives with a form full of questions: *Is there a fire exit? Is the fridge cold enough?* If a box is blank, that is a problem.

Our "form" is a plain Python dictionary with **six sections**. A value of `None` means **"nobody has decided this yet"**. Every `None` will become a *gap* (a problem to fix).

| Section | The question it asks | Restaurant version |
| --- | --- | --- |
| `nfr` | What numbers must we hit? | Speed limit for the kitchen |
| `api_boundary` | Who may come in, and how? | Front door + bouncer |
| `trust_boundaries` | Where do we check things we don't control? | Security at the entrance |
| `state` | What do we remember, where, for how long? | Guest registry |
| `failure_domains` | What happens when a helper stops working? | Backup generator |
| `risks` | What can go wrong, and what stops it? | Fire-safety plan |

### Form 1: the L2 prototype (works on a laptop, lots of blanks)

In [2]:
L2_PROTOTYPE = {
    "name": "SupportDesk (L2 prototype)",
    # 1. Numbers we must hit. All blank: nobody wrote them down.
    "nfr": {"latency_budget_ms": None, "token_budget": None, "mcp_timeout_s": None, "max_tool_calls": None,
            "availability_pct": None, "cost_per_request_usd": None},
    # 2. The front door. All blank: anyone can walk in.
    "api_boundary": {"endpoint": None, "auth": None, "request_schema": None, "max_request_chars": None, "rate_limit_per_min": None},
    # 3. Security checks. All "no".
    "trust_boundaries": {"user_input_validated": False, "model_output_validated": False, "tool_allowlist": []},
    # 4. Memory. Saved only in RAM, so a restart wipes everything.
    "state": {"checkpoint_store": "memory", "thread_id_source": None, "retention_days": None},
    # 5. Helpers that can fail. No timeout, no backup plan.
    "failure_domains": {"mcp_server": {"timeout_s": None, "fallback": None},
                        "model_api": {"timeout_s": None, "fallback": None}},
    # 6. Hazards. Nobody wrote down what stops them.
    "risks": {"model outage": None, "prompt injection": None, "freeze_card without approval": None,
              "duplicate freeze_card": None, "card number in answer": None},
}
print("Prototype form loaded. Blanks are everywhere.")

Prototype form loaded. Blanks are everywhere.


### Form 2: the production target (every box filled in)

This is the **finished plan** for SupportDesk. These are the same names and numbers we will use on Day 19 when we build the whole thing, so keep this form. Every later day builds one box of it.

Read the comments. Each line answers one question from the table above.

In [3]:
PRODUCTION_TARGET = {
    "name": "SupportDesk (production target = Day 19 architecture)",
    # 1. Numbers: every limit is a number we can test.
    "nfr": {"latency_budget_ms": 30000, "token_budget": 200, "mcp_timeout_s": 30, "max_tool_calls": 1,
            "availability_pct": 99.5, "cost_per_request_usd": 0.02},
    # 2. Front door: a key is required, input has a shape, a size limit and a rate limit.
    "api_boundary": {"endpoint": "POST /ask (+ GET /health, /ready, /version)", "auth": "x-api-key header, fail closed",
                     "request_schema": "AskRequest (pydantic)", "max_request_chars": 500, "rate_limit_per_min": 30},
    # 3. Security checks are on, and only one safe tool is allowed without a human.
    "trust_boundaries": {"user_input_validated": True, "model_output_validated": True,
                         "tool_allowlist": ["lookup_balance"]},  # freeze_card needs a human
    # 4. Memory: saved on disk, the SERVER makes the thread_id, old data is deleted after 30 days.
    "state": {"checkpoint_store": "sqlite", "thread_id_source": "server-side: user + cid, never the request body", "retention_days": 30},
    # 5. Every helper has a timeout AND a backup plan.
    "failure_domains": {"mcp_server": {"timeout_s": 30, "fallback": "Service busy, please try again later."},
                        "model_api": {"timeout_s": 20, "fallback": "canned answer from retrieved context"}},
    # 6. Every hazard has something that stops it.
    "risks": {"model outage": "timeout + fallback answer", "prompt injection": "INJECTION regex in guard_input + ROLE_TOOLS",
              "freeze_card without approval": "TOOL_POLICY needs_human=True, check_policy denies",
              "duplicate freeze_card": "idempotency key per card", "card number in answer": "redact() in guard_output"},
}
print("Target form loaded. Every box has an answer.")

Target form loaded. Every box has an answer.


## Step 2: The six checks (one inspector per section)

Each *check* is a small function. It reads one section of the form and writes down a note for every problem it finds. A note is just a piece of text such as `"NFR: 'token_budget' is not a positive number"`.

All six checks have the same shape:
1. Look at **one section** of the spec.
2. Make a **list of problems** (it starts empty).
3. Return the old problems **plus** the new ones.

Below, a tiny helper called `gaps_of` lets us ask a single check, *"how many problems do you find in this form?"*

In [4]:
# Helper: run ONE check on ONE form and return its list of problems.
def gaps_of(check, spec):
    return check({"spec": spec, "gaps": [], "report": ""})["gaps"]


# The "clipboard" that travels through the checks. Each check can read it and add to it.
class GapState(TypedDict):
    spec: dict     # the form being inspected
    gaps: list     # the problems found so far
    report: str    # the final written report

print("Helper ready.")

Helper ready.


### Check 1: NFRs (the speed-limit sign)

**Analogy.** A sign that says *"drive carefully"* cannot be enforced. A sign that says *"maximum 50 km/h"* can. If you cannot measure a target, you can never tell whether you missed it.

**NFR** = *non-functional requirement* = a target written as a **number**. Words like "fast", "reliable" or "cheap" do not count.

This check rejects anything that is not a positive number.

In [5]:
def check_nfrs(state):
    gaps = []
    for key, value in state["spec"]["nfr"].items():
        if not isinstance(value, (int, float)) or value <= 0:  # must be a real number above zero
            gaps.append("NFR: '" + key + "' is not a positive number")
    return {"gaps": state["gaps"] + gaps}


print("Problems in the prototype:", len(gaps_of(check_nfrs, L2_PROTOTYPE)))
print("Problems in the target:   ", len(gaps_of(check_nfrs, PRODUCTION_TARGET)))

Problems in the prototype: 6
Problems in the target:    0


### Check 2: The API boundary (the front door and the bouncer)

**Analogy.** A good club has a door with rules: *"Show your ticket. No bags bigger than this. Only 30 people per minute."* The bouncer refuses people **at the door**, so troublemakers never reach the dance floor.

**API boundary** = the front door of our service. In SupportDesk that door is `POST /ask`. It needs: a key (**auth**), the right shape of data (**request schema**), a **size limit**, and a **rate limit**. Bad requests are refused *before* the agent does any work (or spends any money).

This check flags any rule that is still `None` (not decided).

In [6]:
def check_api_boundary(state):
    gaps = []
    for key, value in state["spec"]["api_boundary"].items():
        if value is None:  # None = nobody decided this rule yet
            gaps.append("API: '" + key + "' is not defined")
    return {"gaps": state["gaps"] + gaps}


print("Problems in the prototype:", len(gaps_of(check_api_boundary, L2_PROTOTYPE)))
print("Problems in the target:   ", len(gaps_of(check_api_boundary, PRODUCTION_TARGET)))

Problems in the prototype: 5
Problems in the target:    0


### Check 3: Trust boundaries (airport security)

**Analogy.** At an airport, everything coming from *outside* (people, bags) is checked before it reaches the plane. You do not skip the check just because the passenger looks friendly.

**Trust boundary** = any place where data moves from something **you do not control** to something **you do**. Check it every time. For SupportDesk there are three:

| Crossing | The security check |
| --- | --- |
| User text → the agent | `guard_input` (blocks tricks like "ignore your rules") |
| The model's choice of tool → the tool | `ROLE_TOOLS` (a list of allowed tools) |
| The agent's answer → the user | `redact` (removes card numbers) |

A surprise for beginners: the **model's own output is also untrusted**. You wrote the prompt, but the model can still pick a tool you did not intend.

In [7]:
def check_trust_boundaries(state):
    gaps = []
    boundaries = state["spec"]["trust_boundaries"]
    if not boundaries["user_input_validated"]:
        gaps.append("TRUST: user input crosses into the agent without validation")
    if not boundaries["model_output_validated"]:
        gaps.append("TRUST: model output reaches tools or users without validation")
    if not boundaries["tool_allowlist"]:
        gaps.append("TRUST: no tool allowlist")
    return {"gaps": state["gaps"] + gaps}


print("Problems in the prototype:", len(gaps_of(check_trust_boundaries, L2_PROTOTYPE)))
print("Problems in the target:   ", len(gaps_of(check_trust_boundaries, PRODUCTION_TARGET)))

Problems in the prototype: 3
Problems in the target:    0


### Check 4: State (the hotel front desk)

**Analogy.** A hotel remembers every guest. Three decisions matter:
- **Where** is the guest book kept? On paper in a safe (durable), or only in the receptionist's head (memory)? If the receptionist goes home, a "memory" hotel forgets everyone.
- **Who picks the room-key number?** The hotel does. If guests could pick their *own* key number, one guest could open another guest's room.
- **How long** do we keep old records? Maybe 30 days, then shred them.

In our agent:

| Hotel | SupportDesk |
| --- | --- |
| Guest book on disk | `checkpoint_store = "sqlite"` (not `"memory"`) |
| Key number made by the hotel | `thread_id` made **by the server** from user + correlation id (`cid`). Never taken from the request. |
| Shred after 30 days | `retention_days = 30` |

If a caller could choose the `thread_id`, they could read **someone else's conversation**.

In [8]:
def check_state_architecture(state):
    gaps = []
    memory = state["spec"]["state"]
    if memory["checkpoint_store"] == "memory":
        gaps.append("STATE: checkpoints live in memory and are lost on restart")
    if memory["thread_id_source"] is None:
        gaps.append("STATE: nobody decides where thread_id comes from (users could read each other's threads)")
    if memory["retention_days"] is None:
        gaps.append("STATE: no retention period for saved state")
    return {"gaps": state["gaps"] + gaps}


print("Problems in the prototype:", len(gaps_of(check_state_architecture, L2_PROTOTYPE)))
print("Problems in the target:   ", len(gaps_of(check_state_architecture, PRODUCTION_TARGET)))

Problems in the prototype: 3
Problems in the target:    0


### Check 5: Failure domains (the fuse box and the backup generator)

**Analogy.** A house has a fuse box so that one broken appliance does not black out the whole street, and a backup generator for when the power company fails. You also do not stay on hold with the power company forever. You hang up after a while and switch to the generator.

- **Failure domain** = a helper that can break on its own (our MCP tool server, the AI model's API).
- **Timeout** = how long we wait before we give up ("hang up the phone").
- **Fallback** = what we do instead ("switch to the generator"). For SupportDesk: reply *"Service busy, please try again later."*

Waiting forever is not a plan. Every outside helper needs **both**.

In [9]:
def check_failure_domains(state):
    gaps = []
    for domain, rules in state["spec"]["failure_domains"].items():
        if rules["timeout_s"] is None:
            gaps.append("FAILURE: " + domain + " call has no timeout")
        if rules["fallback"] is None:
            gaps.append("FAILURE: " + domain + " has no fallback")
    return {"gaps": state["gaps"] + gaps}


print("Problems in the prototype:", len(gaps_of(check_failure_domains, L2_PROTOTYPE)))
print("Problems in the target:   ", len(gaps_of(check_failure_domains, PRODUCTION_TARGET)))

Problems in the prototype: 4
Problems in the target:    0


### Check 6: The risk register (the fire-safety plan)

**Analogy.** A fire-safety plan lists each hazard and what stops it: *"Grease fire → fire blanket. Gas leak → shut-off valve."* A hazard with **no** answer next to it is a hole in the plan.

For SupportDesk, the hazards are things like a model outage, a **prompt injection** (someone tricks the model with sneaky text), freezing a card without a human's approval, freezing the same card twice, and leaking a card number. Each needs a **mitigation** (the thing that stops it).

In [10]:
def check_risk_register(state):
    gaps = []
    for risk, mitigation in state["spec"]["risks"].items():
        if mitigation is None:  # a hazard with no answer is a gap
            gaps.append("RISK: '" + risk + "' has no mitigation")
    return {"gaps": state["gaps"] + gaps}


print("Problems in the prototype:", len(gaps_of(check_risk_register, L2_PROTOTYPE)))
print("Problems in the target:   ", len(gaps_of(check_risk_register, PRODUCTION_TARGET)))

Problems in the prototype: 5
Problems in the target:    0


## Step 3: Put the checks on a conveyor belt (LangGraph)

**Analogy.** Picture a factory belt with seven stations. A clipboard rides along the belt. Each station reads the form, writes its findings on the clipboard, and passes it on. The last station reads the whole clipboard and writes the final report.

- A **node** = one station (one of our functions).
- An **edge** = the belt between two stations.
- The **state** = the clipboard that travels along.

```text
START -> check_nfrs -> check_api_boundary -> check_trust_boundaries -> check_state_architecture
      -> check_failure_domains -> check_risk_register -> write_report -> END
```

The last station, `write_report`, numbers the problems and gives a verdict: **NOT READY** or **READY for review**.

In [11]:
def write_report(state):
    lines = ["Gap report for " + state["spec"]["name"], "Gaps found: " + str(len(state["gaps"]))]
    for number, gap in enumerate(state["gaps"], start=1):
        lines.append(str(number) + ". " + gap)
    verdict = "NOT READY for production" if state["gaps"] else "READY for review"
    lines.append("Verdict: " + verdict)
    return {"report": "\n".join(lines)}


# Build the conveyor belt: list the stations, then connect them in a straight line.
graph = StateGraph(GapState)
checks = [("check_nfrs", check_nfrs), ("check_api_boundary", check_api_boundary),
          ("check_trust_boundaries", check_trust_boundaries), ("check_state_architecture", check_state_architecture),
          ("check_failure_domains", check_failure_domains), ("check_risk_register", check_risk_register),
          ("write_report", write_report)]
for node_name, node_fn in checks:
    graph.add_node(node_name, node_fn)
graph.add_edge(START, "check_nfrs")
graph.add_edge("check_nfrs", "check_api_boundary")
graph.add_edge("check_api_boundary", "check_trust_boundaries")
graph.add_edge("check_trust_boundaries", "check_state_architecture")
graph.add_edge("check_state_architecture", "check_failure_domains")
graph.add_edge("check_failure_domains", "check_risk_register")
graph.add_edge("check_risk_register", "write_report")
graph.add_edge("write_report", END)
app = graph.compile()   # "compile" = close the factory and switch the belt on

print("The conveyor belt is built:", len(checks), "stations.")

The conveyor belt is built: 7 stations.


## Step 4: Run the inspection

We send **two forms** down the same belt, one after the other.

First the **prototype**. Expect many problems. Read them in groups by the word at the start of each line (`NFR`, `API`, `TRUST`, `STATE`, `FAILURE`, `RISK`). Each group matches one check above.

In [12]:
prototype_result = app.invoke({"spec": L2_PROTOTYPE, "gaps": [], "report": ""})
print(prototype_result["report"])

Gap report for SupportDesk (L2 prototype)
Gaps found: 26
1. NFR: 'latency_budget_ms' is not a positive number
2. NFR: 'token_budget' is not a positive number
3. NFR: 'mcp_timeout_s' is not a positive number
4. NFR: 'max_tool_calls' is not a positive number
5. NFR: 'availability_pct' is not a positive number
6. NFR: 'cost_per_request_usd' is not a positive number
7. API: 'endpoint' is not defined
8. API: 'auth' is not defined
9. API: 'request_schema' is not defined
10. API: 'max_request_chars' is not defined
11. API: 'rate_limit_per_min' is not defined
12. TRUST: user input crosses into the agent without validation
13. TRUST: model output reaches tools or users without validation
14. TRUST: no tool allowlist
15. STATE: checkpoints live in memory and are lost on restart
16. STATE: nobody decides where thread_id comes from (users could read each other's threads)
17. STATE: no retention period for saved state
18. FAILURE: mcp_server call has no timeout
19. FAILURE: mcp_server has no fallba

Now the **production target**. The same belt, a different form. This time every box is filled in, so we expect **zero** problems.

In [13]:
target_result = app.invoke({"spec": PRODUCTION_TARGET, "gaps": [], "report": ""})
print(target_result["report"])

Gap report for SupportDesk (production target = Day 19 architecture)
Gaps found: 0
Verdict: READY for review

## Step 5: Where do the gaps hide?

**Analogy.** A good inspector does not just say *"26 problems"*. They say *"6 in the kitchen, 5 at the door, 3 in security..."* so you know where to start.

This cell counts the prototype's problems by group and draws a small bar chart out of text blocks. Later days of the course close these groups one by one.

In [14]:
# The word before the first colon is the group: NFR, API, TRUST, STATE, FAILURE or RISK.
groups = Counter(gap.split(":")[0] for gap in prototype_result["gaps"])
for group in ["NFR", "API", "TRUST", "STATE", "FAILURE", "RISK"]:
    print(f"{group:8} {'#' * groups[group]} {groups[group]}")
print()
print("Total:", sum(groups.values()), "gaps found before writing any production code.")

NFR      ###### 6
API      ##### 5
TRUST    ### 3
STATE    ### 3
FAILURE  #### 4
RISK     ##### 5

Total: 26 gaps found before writing any production code.


That is the whole point of Day 1: finding **26 problems on paper** costs almost nothing. Finding the same problems after launch, with real customers, costs a lot.

## Step 6: Your turn. Predict first, then run

**Why predict?** A pilot reads the checklist *before* flying, not after. If you guess first and then see the answer, your brain remembers it. If you only run the code, you just watch.

We use `copy.deepcopy` to make a private copy of the finished target. That way you can break the copy without touching the original.

### Experiment 1: blank out one number

We take the finished target and set `availability_pct` back to `None`. **How many gaps will the inspection report?** Write your guess down, then open the answer.

<details><summary><b>Click to see the answer</b></summary>

**1 gap.** Only `check_nfrs` looks at that field, and it now says "not a positive number". Every other check still passes.
</details>

In [15]:
broken = copy.deepcopy(PRODUCTION_TARGET)
broken["nfr"]["availability_pct"] = None          # <- the one change

result = app.invoke({"spec": broken, "gaps": [], "report": ""})
print(result["report"])

Gap report for SupportDesk (production target = Day 19 architecture)
Gaps found: 1
1. NFR: 'availability_pct' is not a positive number
Verdict: NOT READY for production

### Experiment 2: also forget the disk

On top of Experiment 1, we change the saved-conversations store from `"sqlite"` (a file on disk) back to `"memory"` (lost on restart). **How many gaps now?**

<details><summary><b>Click to see the answer</b></summary>

**2 gaps.** One from the NFR check (still blank) and one from the STATE check (`checkpoints live in memory and are lost on restart`).
</details>

In [16]:
broken["state"]["checkpoint_store"] = "memory"   # <- second change, on top of the first

result = app.invoke({"spec": broken, "gaps": [], "report": ""})
print(result["report"])

Gap report for SupportDesk (production target = Day 19 architecture)
Gaps found: 2
1. NFR: 'availability_pct' is not a positive number
2. STATE: checkpoints live in memory and are lost on restart
Verdict: NOT READY for production


### Experiment 3 (bonus): invent a new hazard

Real projects find new hazards all the time. Add one to the risk register **without** a mitigation, using a fresh copy of the target. **How many gaps?**

<details><summary><b>Click to see the answer</b></summary>

**1 gap.** A hazard with no mitigation is a gap, no matter how clever the rest of the plan is.
</details>

In [17]:
fresh = copy.deepcopy(PRODUCTION_TARGET)
fresh["risks"]["database disk full"] = None       # a brand-new hazard with no answer yet

result = app.invoke({"spec": fresh, "gaps": [], "report": ""})
print(result["report"])

Gap report for SupportDesk (production target = Day 19 architecture)
Gaps found: 1
1. RISK: 'database disk full' has no mitigation
Verdict: NOT READY for production


**Fix it.** Give the new risk a mitigation, run again, and watch the report go back to zero. (Run the cell below.)

In [18]:
fresh["risks"]["database disk full"] = "alert at 80 percent + cleanup job"

result = app.invoke({"spec": fresh, "gaps": [], "report": ""})
print(result["report"])

Gap report for SupportDesk (production target = Day 19 architecture)
Gaps found: 0
Verdict: READY for review


## Step 7: Assignment (convert an L2 solution into a production target)

You have inspected SupportDesk. Now write the plan for it, on your own. Deliver:

1. **A one-page sketch** of the target architecture: the front door, the model, the tools, the state store.
2. **Three NFRs with numbers** (use the Day 19 names), a **trust-boundary list** with the check at each crossing, a **failure-domain table** (timeout, fallback, where state lives), and a **risk register** of at least three risks.
3. **Both forms** (prototype and target) written as dictionaries in a new notebook cell, plus the two gap reports from the checker. The target must end at **zero gaps**.

Keep your target form. It becomes the Day 19 capstone target, and each later day closes one of its gaps. If you prefer, assess your own L2 Day 20 agent instead, using the same six sections.

## Production checklist (you will reuse these lines on Day 19)
- [ ] **NFRs:** latency, availability and cost-per-request each have a number.
- [ ] **API boundary:** endpoint, auth, schema and max size are defined; bad requests are refused before the graph runs.
- [ ] **State:** saved on disk, `thread_id` comes from the signed-in session, retention is set.
- [ ] **Every outside helper** (model, each tool system) has a timeout and a fallback.
- [ ] **Risk register:** every risk has a mitigation, and the report prints `READY for review`.

## Recap in one minute
| Idea | The analogy | One-line takeaway |
| --- | --- | --- |
| Prototype vs production | Cooking at home vs running a restaurant | The cooking is the same; everything around it is new. |
| NFR | Speed-limit sign | If you cannot measure it, you cannot enforce it. |
| API boundary | Front door and bouncer | Refuse bad visitors before they reach the kitchen. |
| Trust boundary | Airport security | Check anything crossing a line, even the model's own choices. |
| State | Hotel front desk | Keep records on disk, let the hotel pick the key number, shred after 30 days. |
| Failure domain | Fuse box and backup generator | Every outside helper gets a timeout and a fallback. |
| Risk register | Fire-safety plan | A hazard with no answer is a gap. |

## Tomorrow (Day 2)
Today we wrote the plan. Tomorrow we start **building**: a saved conversation that survives a restart, and a pause that waits for a human before a card is frozen.
